# LAB 03 - Regresion Lineal Multivariable y Polinomica
**Dataset:** Communities and Crime Unnormalized (UCI)

**Estudiante:** Sebastian Arduz

**Materia:** SIS420 - Inteligencia Artificial I

**m = 2,214 filas | n = 100 features**

Se predice la tasa de crimenes violentos por cada 100,000 habitantes (ViolentCrimesPerPop) a partir de datos socioeconomicos del censo de EEUU combinados con estadisticas del FBI.

Se comparan tres modelos:
1. Regresion Lineal Multivariable (Gradiente Descendente)
2. Regresion Polinomica (Gradiente Descendente)
3. Ecuacion Normal

## 1. Cargar datos con Pandas

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cols_all = ['communityname','state','countyCode','communityCode','fold',
'population','householdsize','racepctblack','racePctWhite','racePctAsian','racePctHisp',
'agePct12t21','agePct12t29','agePct16t24','agePct65up','numbUrban','pctUrban',
'medIncome','pctWWage','pctWFarmSelf','pctWInvInc','pctWSocSec','pctWPubAsst','pctWRetire',
'medFamInc','perCapInc','whitePerCap','blackPerCap','indianPerCap','AsianPerCap',
'OtherPerCap','HispPerCap','NumUnderPov','PctPopUnderPov','PctLess9thGrade',
'PctNotHSGrad','PctBSorMore','PctUnemployed','PctEmploy','PctEmplManu','PctEmplProfServ',
'PctOccupManu','PctOccupMgmtProf','MalePctDivorce','MalePctNevMarr','FemalePctDiv',
'TotalPctDiv','PersPerFam','PctFam2Par','PctKids2Par','PctYoungKids2Par','PctTeen2Par',
'PctWorkMomYoungKids','PctWorkMom','NumIlleg','PctIlleg','NumImmig','PctImmigRecent',
'PctImmigRec5','PctImmigRec8','PctImmigRec10','PctRecentImmig','PctRecImmig5',
'PctRecImmig8','PctRecImmig10','PctSpeakEnglOnly','PctNotSpeakEnglWell',
'PctLargHouseFam','PctLargHouseOccup','PersPerOccupHous','PersPerOwnOccHous',
'PersPerRentOccHous','PctPersOwnOccup','PctPersDenseHous','PctHousLess3BR','MedNumBR',
'HousVacant','PctHousOccup','PctHousOwnOcc','PctVacantBoarded','PctVacMore6Mos',
'MedYrHousBuilt','PctHousNoPhone','PctWOFullPlumb','OwnOccLowQuart','OwnOccMedVal',
'OwnOccHiQuart','RentLowQ','RentMedian','RentHighQ','MedRent','MedRentPctHousInc',
'MedOwnCostPctInc','MedOwnCostPctIncNoMtg','NumInShelters','NumStreet',
'PctForeignBorn','PctBornSameState','PctSameHouse85','PctSameCity85','PctSameState85',
'LemasSwornFT','LemasSwFTPerPop','LemasSwFTFieldOps','LemasSwFTFieldPerPop',
'LemasTotalReq','LemasTotReqPerPop','PolicReqPerOffic','PolicPerPop',
'RacialMatchCommPol','PctPolicWhite','PctPolicBlack','PctPolicHisp','PctPolicAsian',
'PctPolicMinor','OfficAssgnDrugUnits','NumKindsDrugsSeiz','PolicAveOTWorked',
'LandArea','PopDens','PctUsePubTrans','PolicCars','PolicOperBudg',
'LemasPctPolicOnPatr','LemasGangUnitDeploy','LemasPctOfficDrugUn','PolicBudgPerPop',
'ViolentCrimesPerPop','murders','murdPerPop','rapes','rapesPerPop',
'robberies','robbbPerPop','assaults','assaultPerPop','burglaries','burglPerPop',
'larcenies','larcPerPop','autoTheft','autoTheftPerPop','arsons','arsonsPerPop',
'nonViolPerPop']

datos = pd.read_csv('CommViolPredUnnormalizedData.txt', header=None, names=cols_all, na_values='?')
print(f'Dataset original: {datos.shape[0]} filas, {datos.shape[1]} columnas')

## 2. Preprocesamiento
Se eliminan columnas no predictivas, columnas con muchos valores faltantes, y filas incompletas.

In [ ]:
datos = datos.drop(['communityname','state','countyCode','communityCode','fold'], axis=1)

targets_extra = ['murders','murdPerPop','rapes','rapesPerPop','robberies','robbbPerPop',
                 'assaults','assaultPerPop','burglaries','burglPerPop','larcenies','larcPerPop',
                 'autoTheft','autoTheftPerPop','arsons','arsonsPerPop','nonViolPerPop']
datos = datos.drop(targets_extra, axis=1)

threshold = len(datos) * 0.3
datos = datos.dropna(thresh=len(datos) - int(threshold), axis=1)
datos = datos.dropna()

print(f'Dataset limpio: {datos.shape[0]} filas, {datos.shape[1]} columnas')
print(f'Features: {datos.shape[1] - 1}')
print(f'Target: ViolentCrimesPerPop')
datos.head()

In [ ]:
X = datos.drop('ViolentCrimesPerPop', axis=1).values.astype(float)
y = datos['ViolentCrimesPerPop'].values.astype(float)

m, n = X.shape
print(f'm = {m} muestras')
print(f'n = {n} features')

# Dividir 80% entrenamiento, 20% prueba
np.random.seed(42)
indices = np.random.permutation(m)
corte = int(m * 0.8)
X_train, X_test = X[indices[:corte]], X[indices[corte:]]
y_train, y_test = y[indices[:corte]], y[indices[corte:]]

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba: {X_test.shape[0]} muestras')

## 3. Funciones base
Las mismas funciones del cuadernillo de clase: normalizacion z-score, funcion de costo J(theta), gradiente descendente y ecuacion normal.

In [ ]:
def featureNormalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

def computeCostMulti(X, y, theta):
    m = len(y)
    J = (1 / (2 * m)) * np.sum((np.dot(X, theta) - y) ** 2)
    return J

def gradientDescentMulti(X, y, theta, alpha, num_iters):
    m = len(y)
    J_historial = np.zeros(num_iters)
    for i in range(num_iters):
        theta = theta - (alpha / m) * (np.dot(X, theta) - y).dot(X)
        J_historial[i] = computeCostMulti(X, y, theta)
    return theta, J_historial

def normalEqn(X, y):
    theta = np.dot(np.dot(np.linalg.inv(np.dot(X.T, X)), X.T), y)
    return theta

print('Funciones definidas')

---
## MODELO 1: Regresion Lineal Multivariable
Se usa gradiente descendente con las features originales (sin terminos polinomicos). Cada feature se normaliza con z-score para que esten en la misma escala.

In [ ]:
X_train_norm, mu_lin, sigma_lin = featureNormalize(X_train)
X_train_lin = np.hstack([np.ones((X_train_norm.shape[0], 1)), X_train_norm])

theta_lin = np.zeros(n + 1)
alpha_lin = 0.01
iters_lin = 10000

theta_lin, J_hist_lin = gradientDescentMulti(X_train_lin, y_train, theta_lin, alpha_lin, iters_lin)

print(f'Costo inicial: {J_hist_lin[0]:,.2f}')
print(f'Costo final:   {J_hist_lin[-1]:,.2f}')
print(f'Reduccion:     {((J_hist_lin[0] - J_hist_lin[-1]) / J_hist_lin[0] * 100):.2f}%')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(iters_lin), J_hist_lin, color='#e17055', linewidth=2)
plt.title('Modelo 1: Regresion Lineal - Costo vs Iteracion', fontsize=14)
plt.xlabel('Iteracion')
plt.ylabel('Costo J(theta)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Predicciones Modelo 1
Se realizan predicciones sobre el conjunto de prueba (mas de 100 predicciones).

In [ ]:
X_test_norm_lin = (X_test - mu_lin) / sigma_lin
X_test_lin = np.hstack([np.ones((X_test_norm_lin.shape[0], 1)), X_test_norm_lin])
pred_lin = np.dot(X_test_lin, theta_lin)

error_lin = np.mean(np.abs(pred_lin - y_test))
print(f'Predicciones realizadas: {len(pred_lin)}')
print(f'Error absoluto promedio: {error_lin:,.2f}')

print(f'\nPrimeras 10 predicciones:')
for i in range(10):
    print(f'  Real: {y_test[i]:8.2f}  |  Predicho: {pred_lin[i]:8.2f}  |  Error: {abs(y_test[i]-pred_lin[i]):8.2f}')

---
## MODELO 2: Regresion Polinomica
Se toman las 20 features mas correlacionadas con el target y se agregan sus terminos al cuadrado (X^2), creando features polinomicas de grado 2. Esto permite capturar relaciones no lineales entre las features y el target.

In [ ]:
# Seleccionar top 20 features por correlacion
corr = datos.corr()['ViolentCrimesPerPop'].drop('ViolentCrimesPerPop').abs().sort_values(ascending=False)
top_features = corr.head(20).index.tolist()
print(f'Top 20 features seleccionadas:')
for i, f in enumerate(top_features):
    print(f'  {i+1}. {f} (corr: {corr[f]:.3f})')

cols_idx = [datos.columns.get_loc(f) for f in top_features]
X_train_sel = X_train[:, cols_idx]
X_test_sel = X_test[:, cols_idx]

In [ ]:
# Construir features polinomicas: [X, X^2]
X_train_poly = np.concatenate([X_train_sel, X_train_sel ** 2], axis=1)
X_test_poly = np.concatenate([X_test_sel, X_test_sel ** 2], axis=1)

n_poly = X_train_poly.shape[1]
print(f'Features polinomicas: {n_poly} (20 originales + 20 al cuadrado)')

X_train_poly_norm, mu_poly, sigma_poly = featureNormalize(X_train_poly)
X_train_poly_full = np.hstack([np.ones((X_train_poly_norm.shape[0], 1)), X_train_poly_norm])

theta_poly = np.zeros(n_poly + 1)
alpha_poly = 0.01
iters_poly = 10000

theta_poly, J_hist_poly = gradientDescentMulti(X_train_poly_full, y_train, theta_poly, alpha_poly, iters_poly)

print(f'Costo inicial: {J_hist_poly[0]:,.2f}')
print(f'Costo final:   {J_hist_poly[-1]:,.2f}')
print(f'Reduccion:     {((J_hist_poly[0] - J_hist_poly[-1]) / J_hist_poly[0] * 100):.2f}%')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(iters_poly), J_hist_poly, color='#6c5ce7', linewidth=2)
plt.title('Modelo 2: Regresion Polinomica - Costo vs Iteracion', fontsize=14)
plt.xlabel('Iteracion')
plt.ylabel('Costo J(theta)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Predicciones Modelo 2

In [ ]:
X_test_poly_norm = (X_test_poly - mu_poly) / sigma_poly
X_test_poly_full = np.hstack([np.ones((X_test_poly_norm.shape[0], 1)), X_test_poly_norm])
pred_poly = np.dot(X_test_poly_full, theta_poly)

error_poly = np.mean(np.abs(pred_poly - y_test))
print(f'Predicciones realizadas: {len(pred_poly)}')
print(f'Error absoluto promedio: {error_poly:,.2f}')

print(f'\nPrimeras 10 predicciones:')
for i in range(10):
    print(f'  Real: {y_test[i]:8.2f}  |  Predicho: {pred_poly[i]:8.2f}  |  Error: {abs(y_test[i]-pred_poly[i]):8.2f}')

---
## MODELO 3: Ecuacion Normal
Solucion cerrada que calcula theta directamente con la formula: theta = (X^T * X)^-1 * X^T * y. No necesita normalizar ni elegir alpha ni iterar.

In [ ]:
X_train_normal = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
theta_normal = normalEqn(X_train_normal, y_train)

costo_normal = computeCostMulti(X_train_normal, y_train, theta_normal)
print(f'Costo (ecuacion normal): {costo_normal:,.2f}')

### Predicciones Modelo 3

In [ ]:
X_test_normal = np.hstack([np.ones((X_test.shape[0], 1)), X_test])
pred_normal = np.dot(X_test_normal, theta_normal)

error_normal = np.mean(np.abs(pred_normal - y_test))
print(f'Predicciones realizadas: {len(pred_normal)}')
print(f'Error absoluto promedio: {error_normal:,.2f}')

print(f'\nPrimeras 10 predicciones:')
for i in range(10):
    print(f'  Real: {y_test[i]:8.2f}  |  Predicho: {pred_normal[i]:8.2f}  |  Error: {abs(y_test[i]-pred_normal[i]):8.2f}')

### Grafica de costo - Ecuacion Normal vs Gradiente Descendente
La ecuacion normal llega al costo optimo en un solo paso. El gradiente descendente necesita miles de iteraciones para acercarse.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(iters_lin), J_hist_lin, color='#e17055', linewidth=2, label='Gradiente Descendente Lineal')
plt.axhline(y=costo_normal, color='#00b894', linewidth=2, linestyle='--', label=f'Ecuacion Normal (J={costo_normal:,.0f})')
plt.title('Modelo 3: Ecuacion Normal vs Gradiente Descendente', fontsize=14)
plt.xlabel('Iteracion')
plt.ylabel('Costo J(theta)')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Comparacion de los 3 modelos

In [ ]:
print('=' * 60)
print('COMPARACION DE MODELOS')
print('=' * 60)
print(f'{"Modelo":<35} {"Error Promedio":>15}')
print('-' * 60)
print(f'{"1. Lineal Multivariable (GD)":<35} {error_lin:>15,.2f}')
print(f'{"2. Polinomica grado 2 (GD)":<35} {error_poly:>15,.2f}')
print(f'{"3. Ecuacion Normal":<35} {error_normal:>15,.2f}')
print('-' * 60)

mejor = min(error_lin, error_poly, error_normal)
if mejor == error_lin:
    print('Mejor modelo: Lineal Multivariable')
elif mejor == error_poly:
    print('Mejor modelo: Polinomica')
else:
    print('Mejor modelo: Ecuacion Normal')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, pred_lin, alpha=0.4, color='#e17055', s=15)
axes[0].plot([0, y_test.max()], [0, y_test.max()], 'k--', linewidth=1)
axes[0].set_title(f'Lineal (Error: {error_lin:,.0f})', fontsize=12)
axes[0].set_xlabel('Real')
axes[0].set_ylabel('Predicho')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, pred_poly, alpha=0.4, color='#6c5ce7', s=15)
axes[1].plot([0, y_test.max()], [0, y_test.max()], 'k--', linewidth=1)
axes[1].set_title(f'Polinomica (Error: {error_poly:,.0f})', fontsize=12)
axes[1].set_xlabel('Real')
axes[1].set_ylabel('Predicho')
axes[1].grid(True, alpha=0.3)

axes[2].scatter(y_test, pred_normal, alpha=0.4, color='#00b894', s=15)
axes[2].plot([0, y_test.max()], [0, y_test.max()], 'k--', linewidth=1)
axes[2].set_title(f'Ec. Normal (Error: {error_normal:,.0f})', fontsize=12)
axes[2].set_xlabel('Real')
axes[2].set_ylabel('Predicho')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Real vs Predicho - Comparacion de Modelos', fontsize=14)
plt.tight_layout()
plt.show()

## Grafica de relacion Feature vs Target
Se muestra la feature mas correlacionada con el target, para visualizar la relacion.

In [ ]:
mejor_feature = corr.index[0]
idx_feat = datos.columns.get_loc(mejor_feature)

plt.figure(figsize=(10, 5))
plt.scatter(datos[mejor_feature], datos['ViolentCrimesPerPop'], alpha=0.3, color='#00cec9', s=10)
plt.title(f'{mejor_feature} vs ViolentCrimesPerPop (corr: {corr.iloc[0]:.3f})', fontsize=14)
plt.xlabel(mejor_feature)
plt.ylabel('ViolentCrimesPerPop')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()